In [11]:
TEAM_NAME_TO_ABBR = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BRK",
    "Chicago Bulls": "CHI",
    "Charlotte Hornets": "CHO",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHO",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

def normalize_team(name):
    if name is None:
        return None
    return TEAM_NAME_TO_ABBR.get(name, name)

In [ ]:
import re

def parse_transaction(description):
    desc = description.strip().rstrip(".")

    result_rows = []

    #caso trade
    if " traded " in desc:
        operation_type = "trade"

        has_trade_exception = "trade exception" in desc.lower()
        clean_desc = desc.split(" for ")[0]

        parts = clean_desc.split(";")

        for part in parts:
            part = part.strip()

            if " traded " not in part:
                continue

            team_from = part.split(" traded ")[0].replace("The ", "").strip()
            rest = part.split(" traded ")[1]

            if " to " not in rest:
                continue

            assets_text, team_to = rest.split(" to ", 1)
            team_to = team_to.replace("The ", "").strip()
            assets_text = assets_text.replace(" and ", ", ")
            assets = [a.strip() for a in assets_text.split(",")]

            for asset in assets:
                row = {
                    "operation_type": operation_type,
                    "team_from": team_from,
                    "team_to": team_to,
                    "asset_type": None,
                    "asset_name": asset,
                    "pick_year": None,
                    "pick_round": None,
                    "pick_original_team": None,
                    "has_trade_exception": has_trade_exception
                }

                asset_lower = asset.lower()

                if "draft pick" in asset_lower:
                    row["asset_type"] = "pick"

                    year_match = re.search(r"\b(20\d{2})\b", asset)
                    if year_match:
                        row["pick_year"] = int(year_match.group(1))

                    if "1st" in asset:
                        row["pick_round"] = 1
                    elif "2nd" in asset:
                        row["pick_round"] = 2

                    team_match = re.search(r"is ([A-Z]{3}) own", asset)
                    if team_match:
                        row["pick_original_team"] = team_match.group(1)

                elif "cash" in asset_lower:
                    row["asset_type"] = "cash"
                    row["asset_name"] = "cash"
                else:
                    row["asset_type"] = "player"

                result_rows.append(row)

    #caso sign
    elif " signed " in desc:
        operation_type = "sign"

        team_to = desc.split(" signed ")[0].replace("The ", "").strip()
        player_part = desc.split(" signed ")[1]
        player_name = player_part.split(" to ")[0].strip()

        result_rows.append({
            "operation_type": operation_type,
            "team_from": None,
            "team_to": team_to,
            "asset_type": "player",
            "asset_name": player_name,
            "pick_year": None,
            "pick_round": None,
            "pick_original_team": None,
            "has_trade_exception": False
        })

    #caso waive
    elif " waived " in desc:
        operation_type = "waive"

        team_from = desc.split(" waived ")[0].replace("The ", "").strip()
        player_name = desc.split(" waived ")[1].strip()

        result_rows.append({
            "operation_type": operation_type,
            "team_from": team_from,
            "team_to": None,
            "asset_type": "player",
            "asset_name": player_name,
            "pick_year": None,
            "pick_round": None,
            "pick_original_team": None,
            "has_trade_exception": False
        })

    #outros casos
    else:
        result_rows.append({
            "operation_type": "other",
            "team_from": None,
            "team_to": None,
            "asset_type": None,
            "asset_name": desc,
            "pick_year": None,
            "pick_round": None,
            "pick_original_team": None,
            "has_trade_exception": False
        })

    return result_rows

In [ ]:
from datetime import datetime
import time
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd

START_YEAR = 1950
END_YEAR = 2026
BASE_URL = "https://www.basketball-reference.com/leagues/NBA_{}_transactions.html"

scraper = cloudscraper.create_scraper()

data = []
trade_id = 0


from datetime import datetime

def parse_date(date_raw):
    date_raw = date_raw.strip()

    # caso normal: "February 23, 2026"
    try:
        return datetime.strptime(date_raw, "%B %d, %Y").date().isoformat()
    except ValueError:
        pass

    # caso com dia desconhecido: "August ?, 1949"
    try:
        date_fixed = date_raw.replace("?", "01")
        return datetime.strptime(date_fixed, "%B %d, %Y").date().isoformat()
    except ValueError:
        pass

    # fallback: retorna None se não conseguir interpretar
    return None

parsed_rows = []

for year in range(START_YEAR, END_YEAR + 1):
    url = BASE_URL.format(year)
    print(f"Baixando: {url}")

    response = scraper.get(url)
    response.raise_for_status()

    html = response.text
    html = html.replace("<!--", "").replace("-->", "")

    soup = BeautifulSoup(html, "lxml")

    ul = soup.find("ul", class_="page_index")
    if ul is None:
        print(f"Não foi encontrado <ul class='page_index'> em {year}")
        continue

    for li in ul.find_all("li", recursive=False):
        span = li.find("span")
        if span is None:
            continue

        date_raw = span.get_text(strip=True)
        date = parse_date(date_raw)

        for p in li.find_all("p", recursive=False):

            rows = parse_transaction_html(p)

            for row in rows:
                row["trade_id"] = trade_id
                row["date"] = date

                row["team_from"] = normalize_team(row["team_from"])
                row["team_to"]   = normalize_team(row["team_to"])

                parsed_rows.append(row)
                trade_id += 1

    time.sleep(3)

print(f"Total de linhas após parsing: {len(parsed_rows)}")

df = pd.DataFrame(parsed_rows)
df.to_csv("nba_trades_1950_2026.csv", index=False, encoding="utf-8")
print("Arquivo salvo: nba_trades_1950_2026.csv")

📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1950_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1951_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1952_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1953_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1954_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1955_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1956_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1957_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1958_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1959_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1960_transactions.html
📥 Baixando: https://www.basketball-reference.com/leagues/NBA_1961_transactio

In [ ]:
from bs4 import NavigableString, Tag
import re

def clean_text(txt):
    txt = re.sub(r"\(.*?\)", "", txt)
    return txt.strip()

def is_team_link(tag):
    return isinstance(tag, Tag) and tag.name == "a" and tag.get("href", "").startswith("/teams/")

def is_player_link(tag):
    return isinstance(tag, Tag) and tag.name == "a" and tag.get("href", "").startswith("/players/")

def split_by_semicolon_preserving_tags(p):
    segments = []
    current = []

    for node in p.contents:
        if isinstance(node, NavigableString) and ";" in node:
            parts = node.split(";")
            current.append(parts[0])
            segments.append(current)
            current = [parts[1]]
        else:
            current.append(node)

    if current:
        segments.append(current)

    return segments

def extract_picks(text):
    text = clean_text(text)
    return re.findall(r"\d{4} .*?draft pick", text, flags=re.IGNORECASE)

def parse_trade_segment(nodes):
    team_from = None
    team_to = None
    assets = []

    state = "START"

    for node in nodes:
        if isinstance(node, NavigableString):
            t = clean_text(str(node)).lower()
            if "traded" in t:
                state = "SENDING"
            elif " to " in t:
                state = "TO"

        elif isinstance(node, Tag) and node.name == "a":
            label = node.get_text(strip=True)

            if is_team_link(node):
                if team_from is None:
                    team_from = label
                else:
                    team_to = label

            elif is_player_link(node) and state == "SENDING":
                assets.append(label)

    text = clean_text(" ".join(n.get_text(" ", strip=True) if isinstance(n, Tag) else str(n) for n in nodes))
    picks = extract_picks(text)
    assets.extend(picks)

    return team_from, team_to, assets

def parse_transaction_html(p):
    text = p.get_text(" ", strip=True).lower()
    if "traded" not in text:
        return []

    rows = []

    segments = split_by_semicolon_preserving_tags(p)

    for seg in segments:
        team_from, team_to, assets = parse_trade_segment(seg)

        if not team_from or not team_to:
            continue

        for asset in assets:
            asset_lower = asset.lower()

            if "draft pick" in asset_lower:
                asset_type = "pick"
            elif "cash" in asset_lower:
                asset_type = "cash"
            else:
                asset_type = "player"

            rows.append({
                "operation_type": "trade",
                "team_from": team_from,
                "team_to": team_to,
                "asset_type": asset_type,
                "asset_name": asset,
                "pick_year": None,
                "pick_round": None,
                "pick_original_team": None,
                "has_trade_exception": "trade exception" in text
            })

    return rows